<a href="https://colab.research.google.com/github/Fei-Q/dsst389/blob/main/nb/notebook19a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Notebook19a

In [1]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds-py/refs/heads/main/funs.py

In [2]:

import numpy as np
import polars as pl

from funs import *
from plotnine import *
from polars import col as c
theme_set(theme_minimal())
pl.Config.set_fmt_str_lengths(1000)

ub = "https://raw.githubusercontent.com/taylor-arnold/fds-py-nb/refs/heads/main/"

### Small Example

Much of the code itself is already included in this notebook. We are going to go through it slowly and intentionally to understand how to parse text annotations in Python. To start, load the small English spaCy model:

In [3]:
import spacy
nlp = spacy.load("en_core_web_sm")

Next, create a corpus object with a single text. I've filled in one example, but you should use the string that you had for homework.

In [4]:
docs = pl.DataFrame({
    "doc_id": "Example",
    "text": "All human wisdom is contained in these two words—'Wait and hope.'"
})

Now parse the text with the NLP model.

In [5]:
anno = DSText.process(docs, nlp)
print_rows(anno)

doc_id,sid,tid,token,token_with_ws,lemma,upos,tag,is_alpha,is_stop,is_punct,dep,head_idx,ent_type,ent_iob
str,i64,i64,str,str,str,str,str,bool,bool,bool,str,i64,str,str
"""Example""",1,1,"""All""","""All ""","""all""","""DET""","""DT""",true,true,false,"""det""",3,"""""","""O"""
"""Example""",1,2,"""human""","""human ""","""human""","""ADJ""","""JJ""",true,false,false,"""amod""",3,"""""","""O"""
"""Example""",1,3,"""wisdom""","""wisdom ""","""wisdom""","""NOUN""","""NN""",true,false,false,"""nsubjpass""",5,"""""","""O"""
"""Example""",1,4,"""is""","""is ""","""be""","""AUX""","""VBZ""",true,true,false,"""auxpass""",5,"""""","""O"""
"""Example""",1,5,"""contained""","""contained ""","""contain""","""VERB""","""VBN""",true,false,false,"""ROOT""",5,"""""","""O"""
"""Example""",1,6,"""in""","""in ""","""in""","""ADP""","""IN""",true,true,false,"""prep""",5,"""""","""O"""
"""Example""",1,7,"""these""","""these ""","""these""","""DET""","""DT""",true,true,false,"""det""",9,"""""","""O"""
"""Example""",1,8,"""two""","""two ""","""two""","""NUM""","""CD""",true,true,false,"""nummod""",9,"""CARDINAL""","""B"""
"""Example""",1,9,"""words—'Wait""","""words—'Wait ""","""words—'wait""","""NOUN""","""NN""",false,false,false,"""pobj""",6,"""""","""O"""


Compare this to what you did for the homework. Are there any differences?

### Grabbing Some Data

Next, we are going to grab some text data from Wikipedia. I've written a script that called the Wikipedia API to get the text from a page. Here is the function that takes a page name and returns the text.

In [6]:
import re
import requests
from lxml import html

def wiki_get_text(page):
    resp = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={
            "action": "parse",
            "page": page,
            "prop": "text",
            "format": "json",
        },
        headers={"User-Agent": "MyBot/1.0 (myemail@example.com)"},
    )
    raw_html = resp.json()["parse"]["text"]["*"]
    tree = html.fromstring(raw_html)
    text = " ".join(p.text_content().strip() for p in tree.xpath("//p"))
    text = re.sub(r"\[\d+\]", "", text)

    return text

We can use this inside of a call to `pl.DataFrame` to build a textual corpus object. Here is an example for the data science page. Start with this one and then come back with your own example.

In [21]:
df = pl.DataFrame({
    "doc_id": "Parnassius_hunnyngtoni",
    "text": wiki_get_text("Parnassius_hunnyngtoni")
})
df

doc_id,text
str,str
"""Parnassius_hunnyngtoni""",""" Parnassius hunnyngtoni, the Hannyngton's Apollo, is a high-altitude butterfly which is found in India. It is a member of the snow Apollo genus (Parnassius) of the swallowtail family (Papilionidae). Some sources also spell the name as P. hunnygtoni. It is named after Frank Hannyngton who obtained the specimen from the Chumbi Valley. Northern slope of central Himalaya (Tibet). Previously known from the Chumbi Valley, northern part of India (Sikkim) which currently is occupied by China. It may be in Bhutan. Original description by Andrey Avinoff: I received this wonderful new species through the kindness of Mr. Hunnyngton, in honour of whom I have named this minute Parnassius. It comes from high elevations near the Chumbi Valley, South Tibet. Apparently this new species belongs to the acco-group, as may be seen by the corneous bag sphragis of the female and the characteristic white scaled veins of the slightly pinkish surface of the underside of the secondaries. Their pattern is very pec…"


Once you have the data, you can process it with the NLP annotation:

In [22]:
anno = DSText.process(df, nlp)
anno

doc_id,sid,tid,token,token_with_ws,lemma,upos,tag,is_alpha,is_stop,is_punct,dep,head_idx,ent_type,ent_iob
str,i64,i64,str,str,str,str,str,bool,bool,bool,str,i64,str,str
"""Parnassius_hunnyngtoni""",1,1,""" """,""" """,""" ""","""SPACE""","""_SP""",false,false,false,"""dep""",2,"""""","""O"""
"""Parnassius_hunnyngtoni""",1,2,"""Parnassius""","""Parnassius ""","""Parnassius""","""PROPN""","""NNP""",true,false,false,"""compound""",3,"""""","""O"""
"""Parnassius_hunnyngtoni""",1,3,"""hunnyngtoni""","""hunnyngtoni""","""hunnyngtoni""","""NOUN""","""NN""",true,false,false,"""nsubj""",10,"""""","""O"""
"""Parnassius_hunnyngtoni""",1,4,""",""",""", """,""",""","""PUNCT""",""",""",false,false,true,"""punct""",3,"""""","""O"""
"""Parnassius_hunnyngtoni""",1,5,"""the""","""the ""","""the""","""DET""","""DT""",true,true,false,"""det""",6,"""""","""O"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Parnassius_hunnyngtoni""",38,9,"""by""","""by ""","""by""","""ADP""","""IN""",true,true,false,"""agent""",8,"""""","""O"""
"""Parnassius_hunnyngtoni""",38,10,"""law""","""law ""","""law""","""NOUN""","""NN""",true,false,false,"""pobj""",9,"""""","""O"""
"""Parnassius_hunnyngtoni""",38,11,"""in""","""in ""","""in""","""ADP""","""IN""",true,true,false,"""prep""",8,"""""","""O"""


1. Now, for a little work on your end. Using the code that was in the book, find the top 10 most common nouns on the page.

In [23]:
(
    anno
    .filter(c.upos == "NOUN")
    .group_by(c.lemma)
    .agg(count = pl.len())
    .sort(by = "count", descending = True)
    .head(10)
)

lemma,count
str,u32
"""marking""",9
"""specie""",7
"""spelling""",5
"""basal""",4
"""name""",4
"""band""",4
"""female""",4
"""surface""",3
"""acco""",3


2. And then find the top ten common adjectives.

In [24]:
(
    anno
    .filter(c.upos == "ADJ")
    .group_by(c.lemma)
    .agg(count = pl.len())
    .sort(by = "count", descending = True)
    .head(10)
)

lemma,count
str,u32
"""dark""",8
"""red""",3
"""original""",3
"""curved""",3
"""central""",3
"""high""",2
"""black""",2
"""such""",2
"""latter""",2


Go back and choose another page and see how the outputs change. Then, wait for use to come together for the next step.

### Sheets

Next we are going to create a class dataset. Once that is finished, we will use the code below to read it in.

In [25]:
SHEET_ID = "1yyeOB8AcNvoDhSoMbjEEkaKKw6REXLV-mH4oJpg0qB8"
GID = "0"

url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={GID}"

class_df = pl.read_csv(url)
class_df

state,name,page,party
str,str,str,str
"""Alabama""","""Tommy Tuberville""","""Tommy_Tuberville""","""Republican"""
"""Alabama""","""Katie Britt""","""Katie_Britt""","""Republican"""
"""Alaska""","""Lisa Murkowski""","""Lisa_Murkowski""","""Republican"""
"""Alaska""","""Dan Sullivan""","""Dan_Sullivan_(U.S._senator)""","""Republican"""
"""Arizona""","""Mark Kelly""","""Mark_Kelly""","""Democratic"""
…,…,…,…
"""South Dakota""","""John Thune""","""John_Thune""","""Republican"""
"""Tennessee""","""Marsha Blackburn""","""Marsha_Blackburn""","""Republican"""
"""Tennessee""","""Bill Hagerty""","""Bill_Hagerty""","""Republican"""


Then, we can cycle through the rows of the dataset, putting together the Wikipedia data from each row, and then create a corpus object.

In [26]:

wiki_df = []
for row in class_df.iter_rows(named=True):
    wiki_df.append(pl.DataFrame({
        "doc_id": row['name'],
        "text": wiki_get_text(row['page'])
    }))

wiki_df = pl.concat(wiki_df)
wiki_df

doc_id,text
str,str
"""Tommy Tuberville""",""" Thomas Hawley Tuberville (/ˈtʌbərvɪl/; TUH-bərv-il; born September 18, 1954) is an American politician, and retired college football coach and sports broadcaster who is the senior United States senator from Alabama, a seat he has held since 2021. He is a member of the Republican Party. Before entering politics, Tuberville was the head football coach at Auburn University from 1999 to 2008. He was also the head football coach at the University of Mississippi from 1995 to 1998, Texas Tech University from 2010 to 2012, and the University of Cincinnati from 2013 to 2016. Tuberville won five national coach-of-the-year awards (AP, AFCA, Sporting News, Walter Camp, and Bear Bryant) after Auburn's 13–0 season in 2004, in which Auburn won the Southeastern Conference title and the Sugar Bowl, but was left out of the BCS National Championship Game. He earned his 100th career win in 2007. Tuberville is the only Auburn football coach to beat in-state rival Alabama six consecutive times. In 2015, he…"
"""Katie Britt""",""" Katie Elizabeth Boyd Britt (née Boyd; born February 2, 1982) is an American politician and attorney serving since 2023 as the junior United States senator from Alabama. A member of the Republican Party, Britt is the first woman to be elected to the U.S. Senate from Alabama and the youngest Republican woman to be elected to the Senate. She was president and CEO of the Business Council of Alabama from 2019 to 2021, and served as chief of staff for the previous incumbent, Richard Shelby, from 2016 to 2018. Britt was born Katie Elizabeth Boyd on February 2, 1982, to Julian and Debra Boyd in Enterprise, Alabama. During her youth she worked at her family's business. Her family lived near Fort Rucker in Dale County, Alabama. Her father owned a hardware store and later a boat dealership; her mother owned a dance studio. A graduate of Enterprise High School, Britt was a cheerleader and a valedictorian. After graduating in 2000 she studied political science at the University of Alabama. She was…"
"""Lisa Murkowski""",""" Lisa Ann Murkowski (/mərˈkaʊski/ mər-KOW-ski; born May 22, 1957) is an American attorney and politician serving as the senior United States senator from the state of Alaska, having held the seat since 2002. She is the first woman to represent Alaska in the U.S. Congress and is the Senate's second-most senior Republican woman. Murkowski became dean of Alaska's congressional delegation upon Representative Don Young's death. Murkowski is the daughter of former U.S. senator and governor of Alaska Frank Murkowski. She was appointed to the Senate by her father, who resigned his seat in 2002 to become Alaska's governor. Murkowski became the first Alaskan-born member of Congress and completed her father's unexpired Senate term, which ended in January 2005. Before her appointment to the Senate, she had been a member of the Alaska House of Representatives since 1999. Murkowski ran for and won a full term in 2004 with 48% of the vote. After losing the 2010 Republican primary to Tea Party candida…"
"""Dan Sullivan""",""" Daniel Scott Sullivan (born November 13, 1964) is an American politician, attorney, and Marine Corps veteran serving as the junior United States senator from the state of Alaska since 2015. A member of the Republican Party, Sullivan previously served as the commissioner of the Alaska Department of Natural Resources from 2010 to 2013, and as the Alaska Attorney General from 2009 to 2010. Sullivan grew up in a suburb of Cleveland, Ohio and graduated from Culver Academies in Indiana. He studied economics at Harvard University, then earned joint foreign service and Juris Doctor degrees from Georgetown University. He was on active duty for the United States Marine Corps from 1993 to 1997, 2004 to 2006, and in 2009 and 2013. Between 1997 and 1999, he clerked for judges on the United States Court of Appeals for the Ninth Circuit and the Alaska Supreme Court. He worked as 

As before, we will next create an annotation object.

In [27]:
anno = DSText.process(wiki_df, nlp)
anno

doc_id,sid,tid,token,token_with_ws,lemma,upos,tag,is_alpha,is_stop,is_punct,dep,head_idx,ent_type,ent_iob
str,i64,i64,str,str,str,str,str,bool,bool,bool,str,i64,str,str
"""Tommy Tuberville""",1,1,""" """,""" """,""" ""","""SPACE""","""_SP""",false,false,false,"""dep""",2,"""""","""O"""
"""Tommy Tuberville""",1,2,"""Thomas""","""Thomas ""","""Thomas""","""PROPN""","""NNP""",true,false,false,"""compound""",4,"""PERSON""","""B"""
"""Tommy Tuberville""",1,3,"""Hawley""","""Hawley ""","""Hawley""","""PROPN""","""NNP""",true,false,false,"""compound""",4,"""PERSON""","""I"""
"""Tommy Tuberville""",1,4,"""Tuberville""","""Tuberville ""","""Tuberville""","""PROPN""","""NNP""",true,false,false,"""nsubj""",14,"""PERSON""","""I"""
"""Tommy Tuberville""",1,5,"""(""","""(""","""(""","""PUNCT""","""-LRB-""",false,false,true,"""punct""",4,"""""","""O"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Ted Cruz""",470,10,"""comedy""","""comedy ""","""comedy""","""NOUN""","""NN""",true,false,false,"""compound""",11,"""""","""O"""
"""Ted Cruz""",470,11,"""film""","""film ""","""film""","""NOUN""","""NN""",true,false,false,"""compound""",13,"""""","""O"""
"""Ted Cruz""",470,12,"""Lady""","""Lady ""","""Lady""","""PROPN""","""NNP""",true,false,false,"""compound""",13,"""PERSON""","""B"""


3. Adapting the code show in the text, find the 8 nouns that are the most common on each page and print out the results to have a summary of the themes of each page. What patterns do you see? Are there any challenges? What else might you want to do with this data?

In [28]:
(
    anno
    .filter(c.upos == "NOUN")
    .group_by(c.doc_id, c.lemma)
    .agg(count = pl.len())
    .sort(c.doc_id, c.count, descending=[False, True])
    .group_by(c.doc_id)
    .head(8)
    .group_by(c.doc_id)
    .agg(top_adj = c.lemma.sort().str.join("; "))
)

doc_id,top_adj
str,str
"""Gary Peters""","""%; bill; campaign; district; election; member; senator; state"""
"""Tina Smith""","""%; administration; campaign; election; letter; president; senator; state"""
"""Bill Cassidy""","""%; bill; cassidy; child; election; health; legislation; vote"""
"""Maggie Hassan""","""%; bill; campaign; governor; law; senator; state; year"""
"""Cindy Hyde-Smith""","""%; bill; election; office; senator; state; time; vote"""
…,…
"""Lisa Murkowski""","""%; campaign; election; father; seat; senator; time; vote"""
"""Richard Blumenthal""","""attorney; law; letter; member; senator; service; state; year"""
"""John Thune""","""%; election; leader; race; senator; vote; whip; year"""
